# Assess · Prueba independiente y evaluación de la CLI

In [1]:
import json
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "pyproject.toml").is_file():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
results = root / "results"
def read(name):
    return json.loads((results / name).read_text(encoding="utf-8"))

In [2]:
evaluation = read("evaluation.json")
for result in evaluation["comparison"]:
    test = result["test"]
    print(result["algorithm"], "F1 macro:", test["macro_f1"], "Binario:", test["binary"])
test = next(item["test"] for item in evaluation["comparison"] if item["algorithm"] == evaluation["selected"])
for label, metric in test["by_category"].items():
    print(label, "P/R/F1", *(round(metric[key], 3) for key in ("precision", "recall", "f1")), metric["matrix"])
print("Errores en la partición de prueba:", test["errors"])

logistic F1 macro: 1.0 Binario: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 62, 'matrix': {'tn': 62, 'fp': 0, 'fn': 0, 'tp': 62}}
multinomial_nb F1 macro: 1.0 Binario: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 62, 'matrix': {'tn': 62, 'fp': 0, 'fn': 0, 'tp': 62}}
A01:2025 P/R/F1 1.0 1.0 1.0 {'tn': 116, 'fp': 0, 'fn': 0, 'tp': 8}
A02:2025 P/R/F1 1.0 1.0 1.0 {'tn': 112, 'fp': 0, 'fn': 0, 'tp': 12}
A04:2025 P/R/F1 1.0 1.0 1.0 {'tn': 112, 'fp': 0, 'fn': 0, 'tp': 12}
A05:2025 P/R/F1 1.0 1.0 1.0 {'tn': 116, 'fp': 0, 'fn': 0, 'tp': 8}
A07:2025 P/R/F1 1.0 1.0 1.0 {'tn': 116, 'fp': 0, 'fn': 0, 'tp': 8}
A10:2025 P/R/F1 1.0 1.0 1.0 {'tn': 118, 'fp': 0, 'fn': 0, 'tp': 6}
API10:2023 P/R/F1 1.0 1.0 1.0 {'tn': 112, 'fp': 0, 'fn': 0, 'tp': 12}
API1:2023 P/R/F1 1.0 1.0 1.0 {'tn': 116, 'fp': 0, 'fn': 0, 'tp': 8}
API2:2023 P/R/F1 1.0 1.0 1.0 {'tn': 116, 'fp': 0, 'fn': 0, 'tp': 8}
API4:2023 P/R/F1 1.0 1.0 1.0 {'tn': 116, 'fp': 0, 'fn': 0, 'tp': 8}
API8:2023 P/R/F1 1.0 1.0 1.0 {'t

In [3]:
from lab.assess_projects import assess_projects
for name, result in assess_projects(quiet=True).items():
    print(name, {key: result[key] for key in ("tp", "fp", "fn", "precision", "recall", "f1")})
    print("Falsos positivos:", result["false_positives"])
    print("Falsos negativos:", result["false_negatives"])

demo-microservices {'tp': 11, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Falsos positivos: []
Falsos negativos: []
review-microservices {'tp': 2, 'fp': 2, 'fn': 1, 'precision': 0.5, 'recall': 0.6666666666666666, 'f1': 0.5714285714285714}
Falsos positivos: [('dashboard/src/dashboard.ts', 24, 'A01:2025'), ('dashboard/src/dashboard.ts', 24, 'API1:2023')]
Falsos negativos: [('catalog/app/catalog.py', 18, 'API4:2023')]


![F1 por categoría](../results/figures/test_categories.svg)

La prueba por fragmentos comparte convenciones con los datos de entrenamiento. En el
segundo proyecto un control de acceso con nombre fuera del vocabulario origina un falso
positivo y una operación de exportación sin límite pasa inadvertida en la localización
estática. Los resultados de los proyectos son medidas por etiqueta y ubicación anotada.